In [ ]:
# Install the GitHub CardPy checkout into this notebook's Python environment.
from pathlib import Path
import subprocess
import sys

CARDPY_REPO = Path.home() / "Documents" / "GitHub" / "CarDpy"
if not (CARDPY_REPO / "pyproject.toml").is_file():
    raise FileNotFoundError(f"CardPy checkout not found: {CARDPY_REPO}")

# Equivalent to: cd ~/Documents/GitHub/CarDpy && python -m pip install -e .
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    cwd=str(CARDPY_REPO),
)


In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [ ]:
from pathlib import Path
import os
import sys

# Use the current GitHub checkout rather than Scripts/CarDpy-master.
CARDPY_REPO = Path.home() / 'Documents' / 'GitHub' / 'CarDpy'
if not (CARDPY_REPO / 'cardpy' / '__init__.py').is_file():
    raise FileNotFoundError(f'CardPy checkout not found: {CARDPY_REPO}. Update CARDPY_REPO above.')

# A kernel that already imported legacy CardPy must be restarted first.
for module_name, module in tuple(sys.modules.items()):
    if module_name == 'cardpy' or module_name.startswith('cardpy.'):
        module_file = getattr(module, '__file__', None)
        if module_file and not Path(module_file).resolve().is_relative_to(CARDPY_REPO.resolve()):
            raise RuntimeError('A different CardPy copy is already loaded. Restart the kernel and run this notebook again.')
sys.path.insert(0, str(CARDPY_REPO))
import cardpy
print(f'Using CardPy: {Path(cardpy.__file__).resolve()}')
import glob
from smart_health_config import get_config
# Compatibility process preserves the original sequence and output numbering.
from smart_health_processing import process


In [ ]:
# Shared original settings live in smart_health_config.py.
config = get_config("topup")


In [ ]:
volunteer = 'SCH_vol_06'
distortion = 'TOPUP'
even_odd_mode = False # this is if data was split 
sl = 'Even'#'Odd', 'Even'

main_data_folder = '/Volumes/T7_old/Smart_Health/03_Processed_Data/02_cDTI/NifTi' # processed data folder containing the volunteer's TOPUP results
main_output_folder = '/Volumes/T7_old/Smart_Health/03_Processed_Data/02_cDTI/NifTi' # path to the main output folder where the volunteer folder is in


In [ ]:
out_path = os.path.join(main_output_folder, volunteer, distortion)
if even_odd_mode:
    out_path = os.path.join(main_output_folder, volunteer, distortion + '_' + sl)

study_root = os.path.abspath(os.path.join(main_data_folder, volunteer, distortion))
if even_odd_mode:
    study_root += '_' + sl
inpath = glob.glob(os.path.join(study_root, '*_TOPUP_Final_Data'))[0]
config["nifti_paths"] = {
    "image": os.path.join(inpath, "TOPUP_Corrected.nii"),
    "bvals": os.path.join(inpath, "TOPUP_Corrected.bvals"),
    "bvecs": os.path.join(inpath, "TOPUP_Corrected.bvecs"),
    "header": os.path.join(inpath, "TOPUP_Corrected.header"),
}
# TOPUP resumes after correction: no repeat of Gibbs/rejection/respiratory stages.
config["output"]["directory"] = out_path


In [ ]:
results = process(study_root, config)
